# MiTienda · Dashboard en Colab

Ejecuta las celdas en orden. Usa `MiTienda_Render_Seaborn.zip` de esta entrega. **Resultados verificados:** 1.600 facturas, ingresos de COP 203.465.700 y cero diferencias entre facturas y pedidos. Fuente: `Mi_tienda.xlsx` y sus seis tablas CSV originales; el paquete trae un corte analítico sin los contactos originales.

El HTML descargable ofrece 18 gráficos Plotly interactivos. Aquí, tres gráficos Seaborn se recalculan en Python al cambiar filtros.

## 1. Preparar el entorno y cargar el paquete

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

subidos = files.upload()
nombre = 'MiTienda_Render_Seaborn.zip'
if nombre not in subidos:
    raise FileNotFoundError(f'Sube {nombre} desde la ventana de archivos')
carpeta = Path('/content/mitienda')
carpeta.mkdir(exist_ok=True)
permitidos = {'sales.json', 'index.html', 'app.py', 'requirements.txt',
              'Codigo_MiTienda.txt', 'mitienda_dashboard.py', 'README.md', 'render.yaml'}
with ZipFile(nombre) as paquete:
    for miembro in paquete.infolist():
        if miembro.filename in permitidos and not miembro.is_dir():
            (carpeta / miembro.filename).write_bytes(paquete.read(miembro))
assert {'sales.json', 'index.html', 'app.py'} <= {p.name for p in carpeta.iterdir()}
print('Paquete cargado en', carpeta)

In [ ]:
%pip -q install -r /content/mitienda/requirements.txt ipywidgets

## 2. Comprobar el origen y las cifras

In [ ]:
import sys
sys.path.insert(0, str(carpeta))
from app import VENTAS, imagen_seaborn

facturas = len(VENTAS)
ingresos = sum(fila['venta'] for fila in VENTAS)
assert facturas == 1600 and ingresos == 203465700
print(f'{facturas:,} líneas de venta · COP {ingresos:,} · fuente: Mi_tienda.xlsx / seis CSV')

## 3. Explorar gráficos Seaborn con filtros

In [ ]:
import base64
import ipywidgets as widgets
from IPython.display import SVG, display, clear_output

bloque = widgets.Dropdown(options=[('Todos', 0), ('Bloque 1', 1), ('Bloque 2', 2)], description='Bloque')
familias = ['Todas'] + sorted({x['familia'] for x in VENTAS})
familia = widgets.Dropdown(options=familias, description='Familia')
zonas = ['Todas'] + sorted({x['zona'] for x in VENTAS})
zona = widgets.Dropdown(options=zonas, description='Zona')
busqueda = widgets.Text(placeholder='Nombre o ID del producto', description='Buscar')
salida = widgets.Output()

def actualizar(_=None):
    with salida:
        clear_output(wait=True)
        filas = [x for x in VENTAS
                 if (bloque.value == 0 or x['bloque'] == bloque.value)
                 and (familia.value == 'Todas' or x['familia'] == familia.value)
                 and (zona.value == 'Todas' or x['zona'] == zona.value)
                 and (not busqueda.value or busqueda.value.casefold() in x['producto'].casefold()
                      or busqueda.value.casefold() in str(x['producto_id']).casefold())]
        print(f'{len(filas)} líneas de venta · COP {sum(x["venta"] for x in filas):,}')
        for tipo in ('familias', 'calor', 'dispersion'):
            uri = imagen_seaborn(filas, tipo)
            contenido = base64.b64decode(uri.partition(',')[2]).decode('utf-8')
            display(SVG(data=contenido))

for control in (bloque, familia, zona, busqueda):
    control.observe(actualizar, names='value')
display(widgets.VBox([bloque, familia, zona, busqueda]), salida)
actualizar()

## 4. Descargar la entrega

Descarga el HTML y el `.txt` para presentarlos. La exportación HTML incluye Seaborn como SVG del corte completo; el recálculo en Python se demuestra en la celda anterior y en el servidor del paquete.

In [ ]:
files.download(str(carpeta / 'index.html'))
files.download(str(carpeta / 'Codigo_MiTienda.txt'))

## 5. Compartir

Usa **Compartir** de Colab y da acceso de lectura al profesor. Para reproducirlo deberá subir el ZIP y ejecutar las celdas: los archivos de una sesión Colab no se comparten con el cuaderno. Una URL estable del HTML con Seaborn dinámico se obtiene al desplegar el repositorio `drakcore12/seaborn` en Render.